# Part 4 — Data Visualization & Machine Learning
## Student Performance Analysis & Prediction
This notebook analyses a student performance dataset, produces meaningful visualizations, and builds a machine learning model to predict whether a student will pass or fail.

In [ ]:
# Install any missing libraries
!pip install pandas matplotlib seaborn scikit-learn -q

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print('Libraries loaded successfully!')

## Task 1 — Data Exploration with Pandas (5 marks)

In [ ]:
# Load dataset
df = pd.read_csv('students.csv')

# 1. First 5 rows
print('1. First 5 rows:')
df.head()

In [ ]:
# 2. Shape and data types
print(f'2. Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print('\nData types:')
print(df.dtypes)

In [ ]:
# 3. Summary statistics
print('3. Summary statistics:')
df.describe()

In [ ]:
# 4. Count of students who passed and failed
print('4. Pass/Fail count:')
print(df['passed'].value_counts().rename({1: 'Pass', 0: 'Fail'}))

In [ ]:
# 5. Average score per subject for passing vs failing students
subject_cols = ['math', 'science', 'english', 'history', 'pe']

pass_avg = df[df['passed'] == 1][subject_cols].mean()
fail_avg = df[df['passed'] == 0][subject_cols].mean()

print('5. Average scores per subject:')
print(f'{"Subject":<12} {"Pass Avg":>10} {"Fail Avg":>10}')
print('-' * 34)
for subject in subject_cols:
    print(f'{subject:<12} {pass_avg[subject]:>10.2f} {fail_avg[subject]:>10.2f}')

In [ ]:
# 6. Student with highest overall average across all 5 subjects
df['temp_avg'] = df[subject_cols].mean(axis=1)
top_student = df.loc[df['temp_avg'].idxmax()]
print(f'6. Student with highest overall average:')
print(f'   {top_student["name"]} — Average: {top_student["temp_avg"]:.2f}')
df = df.drop(columns=['temp_avg'])

## Task 2 — Data Visualization with Matplotlib (8 marks)

In [ ]:
# Add avg_score column before plotting
subject_cols = ['math', 'science', 'english', 'history', 'pe']
df['avg_score'] = df[subject_cols].mean(axis=1)
print('avg_score column added.')
df[['name', 'avg_score']].head()

In [ ]:
# Plot 1 - Bar Chart: Average score per subject across all students
subject_means = df[subject_cols].mean()

plt.figure(figsize=(8, 5))
plt.bar(subject_means.index, subject_means.values, color='steelblue', edgecolor='black')
plt.title('Average Score per Subject Across All Students')
plt.xlabel('Subject')
plt.ylabel('Average Score')
plt.ylim(0, 100)
for i, val in enumerate(subject_means.values):
    plt.text(i, val + 1, f'{val:.1f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('plot1_bar.png')
plt.show()
print('plot1_bar.png saved.')

In [ ]:
# Plot 2 - Histogram: Distribution of math scores with mean line
mean_math = df['math'].mean()

plt.figure(figsize=(8, 5))
plt.hist(df['math'], bins=5, color='coral', edgecolor='black')
plt.axvline(mean_math, color='navy', linestyle='--', linewidth=2,
            label=f'Mean: {mean_math:.1f}')
plt.title('Distribution of Math Scores')
plt.xlabel('Math Score')
plt.ylabel('Number of Students')
plt.legend()
plt.tight_layout()
plt.savefig('plot2_histogram.png')
plt.show()
print('plot2_histogram.png saved.')

In [ ]:
# Plot 3 - Scatter Plot: study_hours_per_day vs avg_score coloured by passed
pass_df = df[df['passed'] == 1]
fail_df = df[df['passed'] == 0]

plt.figure(figsize=(8, 5))
plt.scatter(pass_df['study_hours_per_day'], pass_df['avg_score'],
            color='green', label='Pass', s=80, edgecolors='black')
plt.scatter(fail_df['study_hours_per_day'], fail_df['avg_score'],
            color='red', label='Fail', s=80, edgecolors='black')
plt.title('Study Hours per Day vs Average Score')
plt.xlabel('Study Hours per Day')
plt.ylabel('Average Score')
plt.legend()
plt.tight_layout()
plt.savefig('plot3_scatter.png')
plt.show()
print('plot3_scatter.png saved.')

In [ ]:
# Plot 4 - Box Plot: attendance_pct for Pass vs Fail students
pass_attendance = df[df['passed'] == 1]['attendance_pct'].tolist()
fail_attendance = df[df['passed'] == 0]['attendance_pct'].tolist()

plt.figure(figsize=(7, 5))
plt.boxplot([pass_attendance, fail_attendance], labels=['Pass', 'Fail'],
            patch_artist=True,
            boxprops=dict(facecolor='lightblue', color='navy'),
            medianprops=dict(color='red', linewidth=2))
plt.title('Attendance % Distribution: Pass vs Fail')
plt.xlabel('Result')
plt.ylabel('Attendance Percentage')
plt.tight_layout()
plt.savefig('plot4_boxplot.png')
plt.show()
print('plot4_boxplot.png saved.')

In [ ]:
# Plot 5 - Line Plot: math and science scores for every student
plt.figure(figsize=(12, 5))
plt.plot(df['name'], df['math'], marker='o', linestyle='-',
         color='steelblue', label='Math')
plt.plot(df['name'], df['science'], marker='s', linestyle='--',
         color='coral', label='Science')
plt.title('Math and Science Scores per Student')
plt.xlabel('Student Name')
plt.ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig('plot5_line.png')
plt.show()
print('plot5_line.png saved.')

## Task 3 — Data Visualization with Seaborn (4 marks)

In [ ]:
# Plot 6 - Seaborn bar plot: avg math and science scores split by passed
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sns.barplot(data=df, x='passed', y='math', ax=ax1, palette='Set2')
ax1.set_title('Average Math Score by Pass/Fail')
ax1.set_xlabel('Passed (0 = Fail, 1 = Pass)')
ax1.set_ylabel('Average Math Score')

sns.barplot(data=df, x='passed', y='science', ax=ax2, palette='Set1')
ax2.set_title('Average Science Score by Pass/Fail')
ax2.set_xlabel('Passed (0 = Fail, 1 = Pass)')
ax2.set_ylabel('Average Science Score')

plt.suptitle('Average Math and Science Scores by Pass/Fail Status', fontsize=13)
plt.tight_layout()
plt.savefig('plot6_seaborn_bar.png')
plt.show()
print('plot6_seaborn_bar.png saved.')

In [ ]:
# Plot 7 - Seaborn scatter with regression lines: attendance_pct vs avg_score
plt.figure(figsize=(9, 6))
sns.regplot(data=df[df['passed'] == 1], x='attendance_pct', y='avg_score',
            label='Pass', color='green', scatter_kws={'edgecolors': 'black'})
sns.regplot(data=df[df['passed'] == 0], x='attendance_pct', y='avg_score',
            label='Fail', color='red', scatter_kws={'edgecolors': 'black'})
plt.title('Attendance % vs Average Score (with Regression Lines)')
plt.xlabel('Attendance Percentage')
plt.ylabel('Average Score')
plt.legend()
plt.tight_layout()
plt.savefig('plot7_seaborn_scatter.png')
plt.show()
print('plot7_seaborn_scatter.png saved.')

# Seaborn vs Matplotlib comparison:
# Seaborn was much easier for grouped statistical plots — sns.barplot() automatically
# computed group means and confidence intervals with a single line of code, whereas
# Matplotlib required manual grouping and calculation. However, Matplotlib gave more
# fine-grained control over individual plot elements like colours, line styles, and
# annotations, which required significantly more effort in Seaborn.

## Task 4 — Machine Learning with scikit-learn (8 marks)

In [ ]:
# Step 1 - Prepare Data
feature_cols = ['math', 'science', 'english', 'history', 'pe',
                'attendance_pct', 'study_hours_per_day']
X = df[feature_cols]
y = df['passed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')

In [ ]:
# Step 2 - Train LogisticRegression model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

train_accuracy = model.score(X_train_scaled, y_train)
print(f'Training Accuracy: {train_accuracy * 100:.2f}%')

In [ ]:
# Step 3 - Evaluate the model
y_pred        = model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')

print('\nPredictions on test set:')
test_names = df.loc[X_test.index, 'name']
for name, actual, predicted in zip(test_names, y_test, y_pred):
    actual_label    = 'Pass' if actual == 1 else 'Fail'
    predicted_label = 'Pass' if predicted == 1 else 'Fail'
    result          = '✅ Correct' if actual == predicted else '❌ Wrong'
    print(f'  {name:<10} | Actual: {actual_label:<4} | '
          f'Predicted: {predicted_label:<4} | {result}')

In [ ]:
# Step 4 - Feature Importance
coefficients        = list(zip(feature_cols, model.coef_[0]))
coefficients_sorted = sorted(coefficients, key=lambda x: abs(x[1]), reverse=True)

print('Feature Coefficients (sorted by absolute value):')
print(f'{"Feature":<25} {"Coefficient":>12}')
print('-' * 38)
for feature, coef in coefficients_sorted:
    direction = '→ Pass' if coef > 0 else '→ Fail'
    print(f'{feature:<25} {coef:>12.4f}  {direction}')

# Horizontal bar chart of feature coefficients
features = [c[0] for c in coefficients_sorted]
coefs    = [c[1] for c in coefficients_sorted]
colors   = ['green' if c > 0 else 'red' for c in coefs]

plt.figure(figsize=(10, 6))
plt.barh(features, coefs, color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.title('Logistic Regression Feature Coefficients\n'
          '(Green = pushes towards Pass, Red = pushes towards Fail)')
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('plot8_feature_importance.png')
plt.show()
print('plot8_feature_importance.png saved.')

In [ ]:
# Step 5 (Bonus) - Predict for a new student
new_student        = [[75, 70, 68, 65, 80, 82, 3.2]]
new_student_scaled = scaler.transform(new_student)
prediction         = model.predict(new_student_scaled)[0]
probability        = model.predict_proba(new_student_scaled)[0]

result_label = 'Pass' if prediction == 1 else 'Fail'
print(f'New student features : {new_student[0]}')
print(f'Prediction           : {result_label}')
print(f'Probability of Fail  : {probability[0]:.4f} ({probability[0]*100:.1f}%)')
print(f'Probability of Pass  : {probability[1]:.4f} ({probability[1]*100:.1f}%)')